# Training phase

## Setting up the configuration
Fisrt let's check the configuration parameters in case you want to change some of them base on the previous analysis. In any case the default configuration should **provide you with a good start**.


In [ ]:
from pathlib import Path
from planktonclas import config, paths

def find_project_config(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, start.parent]
    for root in candidates:
        conf_path = root / 'config.yaml'
        if conf_path.exists():
            return conf_path
    for root in [start, *start.parents]:
        conf_path = root / 'config.yaml'
        if conf_path.exists():
            return conf_path
    raise FileNotFoundError('Could not find config.yaml in the current project. Start Jupyter in the project root or its notebooks folder.')

CONF_PATH = find_project_config()
PROJECT_DIR = CONF_PATH.parent
config.set_config_path(str(CONF_PATH))
paths.CONF = config.get_conf_dict()
print(f'Using project config: {CONF_PATH}')

config.print_conf_table()


In case you want to change the configuration feel free to do so by modifying a project-local `config.yaml` generated by `planktonclas init`, or by adapting the packaged template in `planktonclas/resources/config.yaml`.

## Training

Once you have set up the configuration if needed you can launch the training by running the file `train_runfile.py`


In [ ]:
from datetime import datetime
from planktonclas import config
from planktonclas.train_runfile import train_fn

CONF = config.get_conf_dict()
CONF['dataset']['num_workers'] = 1
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
print(f'Starting training run: {TIMESTAMP}')
train_fn(TIMESTAMP=TIMESTAMP, CONF=CONF)
print(f'Training finished. Use TIMESTAMP = {TIMESTAMP} in the downstream notebooks if you want to inspect this run.')


#### Tips: 
* Running a python file is also done by 
    1. opening a terminal 
    2. typing: `planktonclas train --config ./my_project/config.yaml` or running the training cell in this notebook from the copied project folder
* If you are running several times different training configurations but with the **same** dataset you might consider to add the `mean_RGB` and `std_RGB` parameters to your `config.yaml` file. You can find them computed in any of your already trained models by looking into `../models/[timestamp]/conf/conf.txt`.

## Visualizing the training results

### Display a single timestamp

In [ ]:
import os
import json
from datetime import timedelta

import matplotlib.pylab as plt
import numpy as np

from planktonclas import paths, plot_utils
from datetime import datetime

timestamp = datetime.now().strftime('%Y-%m-%d_%H:%M:%S')
TIMESTAMP = timestamp
# User parameters to set
TIMESTAMP = 'Phytoplankton_EfficientNetV2B0'                       # timestamp of the model

# Set the timestamp
paths.timestamp = TIMESTAMP

# Load training statistics
stats_path = os.path.join(paths.get_stats_dir(), 'stats.json')
with open(stats_path) as f:
    stats = json.load(f)

# Load training configuration
conf_path = os.path.join(paths.get_conf_dir(), 'conf.json')
with open(conf_path) as f:
    conf = json.load(f)

# Plot the trainig plots 
plot_utils.training_plots(conf, stats)

# Print total training time
tr_time = int(stats['training time (s)'])
print('Total training time: {}'.format(timedelta(seconds=tr_time)))

### Compare between multiple timestamps

In [ ]:
from planktonclas.plot_utils import multi_training_plots

TIMESTAMPS = ['Phytoplankton_EfficientNetV2B0',
              'phytoplankton_vliz']

multi_training_plots(timestamps=TIMESTAMPS)